# LLM-Compressor 소개 🗜️

[llm-compressor](https://github.com/vllm-project/llm-compressor)는 vLLM 프로젝트에서 만든 프로덕션용 양자화 도구입니다. 학습 후 양자화(post-training quantization)를 적용할 수 있는 간단한 `oneshot` API를 제공합니다.

이 노트북에서는 다음을 진행합니다.
1. llm-compressor 설정하기
2. `oneshot` API로 작은 모델 양자화하기
3. 양자화 전후의 파일 크기 비교하기

아래 예제는 GPU가 아닌 CPU에서 실행되도록 설계되었습니다. 압축 과정을 보여 주기 위한 간단한 데모일 뿐입니다. 실제 엔터프라이즈 사용 사례의 압축에서는 훨씬 큰 모델을 다루며 여러 개의 GPU가 필요합니다.

## 0. 설정

의존성을 설치합니다. 이 노트북은 CPU에서 실행됩니다.

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
OUTPUT_DIR = "qwen2-0.5b-W4A16"
print("Setup complete.")

In [ ]:
import pathlib
from transformers import AutoTokenizer, AutoModelForCausalLM


---
## 1. Oneshot API

`oneshot` 함수는 llm-compressor의 주요 양자화 인터페이스입니다.

```python
oneshot(
    model="model-name",           # HuggingFace model ID
    dataset="dataset-name",       # Calibration dataset
    recipe=recipe,                # Quantization configuration
    output_dir="./output",        # Where to save
    num_calibration_samples=512,  # Samples for calibration
    max_seq_length=4096,          # Sequence length
)
```

---
## 2. Recipe 이해하기

**recipe**는 llm-compressor가 모델을 어떻게 양자화할지 정의합니다. 양자화 알고리즘과 설정을 지정하는 "modifier"의 목록입니다.

### 사용 가능한 Modifier

llm-compressor는 다양한 양자화 방식에 맞는 여러 modifier를 제공합니다.

**양자화 Modifier:**

| Modifier | 설명 |
|----------|-------------|
| `GPTQModifier` | GPTQ 알고리즘 - 보정(calibration) 데이터를 사용해 최적의 양자화 값을 찾습니다 |
| `AWQModifier` | Activation-aware Weight Quantization - 중요한 가중치를 보존합니다 |
| `AutoRoundModifier` | 학습 가능한 반올림/클리핑을 사용하는 Intel의 알고리즘 |
| `QuantizationModifier` | 단순한 사용 사례를 위한 기본 PTQ 및 QAT |

**변환(Transform) Modifier**(정확도 향상용):

| Modifier | 설명 |
|----------|-------------|
| `SmoothQuantModifier` | 양자화 전에 활성화 값을 평탄화합니다 - GPTQ와 함께 쓰는 경우가 많습니다 |
| `QuIPModifier` | Hadamard 회전으로 이상치를 줄입니다 |
| `SpinQuantModifier` | SpinQuant 방식의 회전으로 가중치 분포를 고르게 만듭니다 |

Modifier는 여러 개를 연결해 사용할 수 있습니다. 예를 들어 `GPTQModifier` 앞에 `SmoothQuantModifier`를 적용하면 W8A8 양자화의 정확도를 높일 수 있습니다.

### GPTQModifier

가장 흔히 쓰이는 modifier는 `GPTQModifier`입니다. 이 modifier는 보정 데이터를 사용해 정확도 손실을 최소화하는 학습 후 양자화 방법인 GPTQ 알고리즘을 사용합니다.

```python
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = [
    GPTQModifier(
        scheme="W4A16",       # Quantization scheme
        targets="Linear",    # Which layers to quantize
        ignore=["lm_head"],  # Layers to skip
        group_size=128,      # Optional: weights per scale factor
    ),
]
```

### 양자화 스킴(Scheme)

`scheme` 파라미터는 가중치(W)와 활성화 값(A)의 비트 수를 결정합니다.

| Scheme | 가중치 | 활성화 값 | 크기 감소 | 품질 영향 |
|--------|---------|-------------|----------------|----------------|
| `W8A16` | 8-bit | 16-bit (FP16) | ~50% | 최소 |
| `W4A16` | 4-bit | 16-bit (FP16) | ~75% | 낮음~보통 |
| `W8A8` | 8-bit | 8-bit | ~50% | 낮음 |
| `W4A8` | 4-bit | 8-bit | ~75% | 보통 |

### 주요 파라미터

| 파라미터 | 용도 | 권장 사항 |
|-----------|---------|----------------|
| `scheme` | 가중치/활성화 값의 비트 수 | 크기와 품질의 균형이 좋은 `W4A16`으로 시작하세요 |
| `targets` | 양자화할 레이어 유형 | `"Linear"`는 모든 linear 레이어를 대상으로 합니다 |
| `ignore` | 원래 정밀도로 유지할 레이어 | 출력 레이어는 민감하므로 항상 `["lm_head"]`를 포함하세요 |
| `group_size` | 하나의 scale factor를 공유하는 가중치 수 | `128`(기본값)이 잘 작동합니다. 작을수록 품질은 좋아지지만 모델이 커집니다 |

### 예제 설정을 이렇게 한 이유

```python
GPTQModifier(
    scheme="W4A16",       # 4-bit weights give ~75% size reduction
    targets="Linear",     # Quantize all linear layers (most of the model)
    ignore=["lm_head"],   # Keep output layer precise for better predictions
)
```

- **W4A16**: 가중치는 공격적으로 압축하되, 안정성을 위해 활성화 값은 FP16으로 유지합니다
- **targets="Linear"**: linear 레이어에 대부분의 파라미터가 있으므로, 이를 양자화하면 가장 크게 절약할 수 있습니다
- **ignore=["lm_head"]**: 마지막 출력 레이어는 어휘 확률로 매핑되므로, FP16으로 유지하면 출력 품질이 보존됩니다

---
## 3. 모델 양자화하기

이제 recipe를 적용해 `Qwen/Qwen2-0.5B-Instruct`를 양자화해 보겠습니다.

In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = [
    GPTQModifier(
        scheme="W4A16",      # 버전에 따라 "INT4" / "W4A8"도 가능
        targets="Linear",
        ignore=["lm_head"],
        # 선택 사항: group_size=128, act_order=True 등,
        # llmcompressor가 제공하는 옵션에 따라 다름
    ),
]

oneshot(
    model=MODEL_ID,
    dataset="wikitext",
    dataset_config_name="wikitext-2-raw-v1",
    recipe=recipe,
    output_dir=OUTPUT_DIR,
    max_seq_length=64,
    num_calibration_samples=4,
)

print(f"\nQuantized model saved to: {OUTPUT_DIR}")

---
## 4. 파일 크기 비교하기

양자화된 모델의 크기를 원본과 비교해 보겠습니다.

In [ ]:
def folder_size(path: str) -> int:
    """디렉터리 안 파일의 전체 크기를 구합니다"""
    p = pathlib.Path(path)
    if not p.exists():
        return 0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def format_size(bytes: int) -> str:
    """바이트 수를 읽기 쉬운 문자열로 변환합니다"""
    if bytes < 1024:
        return f"{bytes} B"
    elif bytes < 1024**2:
        return f"{bytes/1024:.1f} KB"
    elif bytes < 1024**3:
        return f"{bytes/1024**2:.1f} MB"
    return f"{bytes/1024**3:.2f} GB"

In [ ]:
# 양자화된 모델 크기 구하기
size_q = folder_size(OUTPUT_DIR)

print("Model Size Comparison")
print("=" * 40)
print(f"Original (FP16):    ~1.0 GB")
print(f"Quantized (W8A16):   {format_size(size_q)}")

In [ ]:
# 양자화된 모델 파일 목록 보기
!ls -lh {OUTPUT_DIR}/

---
## 5. 모델 테스트하기

원본 모델과 양자화된 모델을 비교하기 위해 간단한 추론 테스트를 실행해 보겠습니다.

In [ ]:
# 원본 모델 불러오기
from transformers import AutoModelForCausalLM, AutoTokenizer

original_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
original_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="cpu")

# 올바른 형식을 위해 chat template을 사용한 테스트 프롬프트
prompt = "Complete this sentence: Model compression is good because"
messages = [{"role": "user", "content": prompt}]
inputs = original_tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)
outputs = original_model.generate(inputs, max_new_tokens=100, do_sample=True, temperature=0.7, pad_token_id=original_tokenizer.eos_token_id)

# 생성된 부분만 디코딩(입력 토큰은 건너뜀)
generated_tokens = outputs[0][inputs.shape[-1]:]
print("Original Model")
print("Prompt:", prompt)
print("Response:", original_tokenizer.decode(generated_tokens, skip_special_tokens=True))

**참고: 다음 섹션에서 양자화된 모델을 불러오는 데 1분 이상 걸릴 수 있습니다**

In [ ]:
# 양자화된 모델 불러오기
import logging
logging.getLogger("llmcompressor").setLevel(logging.WARNING)

quantized_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
quantized_model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR, device_map="cpu")

# 같은 프롬프트로 테스트
inputs = quantized_tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)
outputs = quantized_model.generate(inputs, max_new_tokens=100, do_sample=True, temperature=0.7, pad_token_id=quantized_tokenizer.eos_token_id)

# 생성된 부분만 디코딩(입력 토큰은 건너뜀)
generated_tokens = outputs[0][inputs.shape[-1]:]
print("Quantized Model")
print("Prompt:", prompt)
print("Response:", quantized_tokenizer.decode(generated_tokens, skip_special_tokens=True))

---
## 6. Perplexity 비교

Perplexity(혼란도)는 모델이 텍스트를 얼마나 잘 예측하는지 측정하며, 낮을수록 좋습니다. 원본 모델과 양자화된 모델을 비교해 보겠습니다.

In [ ]:
import math
import torch
from datasets import load_dataset

def calculate_perplexity(model, tokenizer, dataset, max_tokens=5000, stride=512):
    """슬라이딩 윈도 방식으로 perplexity를 계산합니다."""
    # 전체 데이터셋을 하나의 긴 시퀀스로 토큰화
    encodings = tokenizer("\n\n".join(dataset["text"]), return_tensors="pt")
    input_ids = encodings.input_ids[:, :max_tokens]
    
    nlls = []
    prev_end = 0
    
    for begin_loc in range(0, input_ids.size(1), stride):
        end_loc = min(begin_loc + stride, input_ids.size(1))
        trg_len = end_loc - prev_end
        input_slice = input_ids[:, begin_loc:end_loc]
        target_slice = input_slice.clone()
        target_slice[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_slice, labels=target_slice)
            neg_log_likelihood = outputs.loss * trg_len

        nlls.append(neg_log_likelihood)
        prev_end = end_loc

    return math.exp(torch.stack(nlls).sum() / prev_end)

# 테스트 데이터셋 불러오기
test_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

In [ ]:
# 원본 모델의 perplexity 계산
original_ppl = calculate_perplexity(original_model, original_tokenizer, test_dataset)
print(f"Original Model Perplexity: {original_ppl:.2f}")

In [ ]:
# 양자화된 모델의 perplexity 계산
quantized_ppl = calculate_perplexity(quantized_model, quantized_tokenizer, test_dataset)
print(f"Quantized Model Perplexity: {quantized_ppl:.2f}")

In [ ]:
# 결과 비교
print("Perplexity Comparison")
print("=" * 40)
print(f"Original Model:   {original_ppl:.2f}")
print(f"Quantized Model:  {quantized_ppl:.2f}")
print(f"Difference:       {quantized_ppl - original_ppl:+.2f} ({(quantized_ppl/original_ppl - 1)*100:+.1f}%)")

---
## 요약

배운 내용:

✅ `oneshot` API는 llm-compressor의 주요 양자화 인터페이스입니다

✅ `GPTQModifier`로 양자화 알고리즘과 스킴을 설정합니다

✅ 주요 파라미터: `scheme`, `targets`, `ignore`, `group_size`

✅ 양자화하면 모델 크기가 크게 줄어듭니다

✅ 양자화된 모델도 원본과 비슷한 출력을 생성합니다

✅ Perplexity 테스트로 양자화가 품질에 미치는 영향을 측정합니다
